## SHAP Summary Plot — Deep Understanding
What a SHAP Summary Plot answers

Across the entire dataset, which features matter most, and how do their values influence predictions?

It combines:

Global feature importance

Direction of impact

Distribution of effects

Interaction hints

In [ ]:
# ============================================================
# SHAP Summary Plot — FULL ONE-CELL SCRIPT
# ============================================================

# (1) Install SHAP (run once in Colab / fresh env)
!pip -q install shap

In [ ]:
# (2) Imports
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)

# ------------------------------------------------------------
# (3) Dataset
# ------------------------------------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ------------------------------------------------------------
# (4) Train Model
# ------------------------------------------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ------------------------------------------------------------
# (5) SHAP Explainer
# ------------------------------------------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# NOTE:
# shap_values is a list for classification
# Index 1 corresponds to positive class

# ------------------------------------------------------------
# (6) SHAP Summary Plot — Beeswarm (MOST IMPORTANT)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values[1],
    X_test,
    plot_type="dot",
    max_display=15
)

# ------------------------------------------------------------
# (7) SHAP Summary Plot — Bar (Global Importance)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values[1],
    X_test,
    plot_type="bar",
    max_display=15
)

# ------------------------------------------------------------
# (8) Class-wise Comparison (Optional Insight)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar",
    class_names=["Benign", "Malignant"]
)

# ------------------------------------------------------------
# (9) Mean Absolute SHAP (Manual Computation)
# ------------------------------------------------------------
mean_abs_shap = np.abs(shap_values[1]).mean(axis=0)

shap_df = pd.DataFrame({
    "Feature": X.columns,
    "Mean(|SHAP|)": mean_abs_shap
}).sort_values("Mean(|SHAP|)", ascending=False)

print(shap_df.head(10))
